# 🦙 Ollama Server on Colab + ngrok Tunnel

This notebook:
1. Installs **Ollama** on the Colab GPU instance
2. Pulls `gpt-oss:20b` (LLM) and `bge-m3` (embedding)
3. Starts Ollama as a server
4. Creates a **public ngrok URL** — paste this into your `.env` file

**Prerequisites:** Add your `NGROK_AUTH_TOKEN` to Colab Secrets (🔑 icon on the left).

> 🔗 Get a free ngrok token at https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# ── 0. Check GPU ────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── 1. Install Ollama ───────────────────────────────────────────────
!curl -fsSL https://ollama.com/install.sh | sh
print('✅ Ollama installed')

In [ ]:
# ── 2. Start Ollama server in background ────────────────────────────
import subprocess, time

proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)  # wait for server to start

# Verify it's running
import urllib.request
try:
    urllib.request.urlopen('http://localhost:11434')
    print('✅ Ollama server running on http://localhost:11434')
except Exception as e:
    print(f'❌ Server not ready: {e}')

In [ ]:
# ── 3. Pull Models ──────────────────────────────────────────────────
# This may take 5-15 minutes depending on Colab's internet speed

print('📥 Pulling bge-m3 (embedding model, ~580MB)...')
!ollama pull bge-m3
print('✅ bge-m3 ready')

print('\n📥 Pulling gpt-oss:20b (LLM, ~11GB)...')
print('⚠️  This will take a while on first run...')
!ollama pull gpt-oss:20b
print('✅ gpt-oss:20b ready')

In [ ]:
# ── 4. Verify Models ────────────────────────────────────────────────
!ollama list

# Quick smoke test for embedding
import json, urllib.request

data = json.dumps({'model': 'bge-m3', 'input': 'مرحبا'}).encode()
req  = urllib.request.Request(
    'http://localhost:11434/v1/embeddings',
    data=data, headers={'Content-Type': 'application/json'}
)
resp = json.loads(urllib.request.urlopen(req).read())
print(f'✅ bge-m3 embedding dim: {len(resp["data"][0]["embedding"])}')

In [ ]:
# ── 5. Start ngrok Tunnel ───────────────────────────────────────────
!pip install pyngrok -q

from pyngrok import ngrok, conf
from google.colab import userdata

# Get ngrok token from Colab Secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Open tunnel to Ollama port
tunnel = ngrok.connect(11434, proto='http')
PUBLIC_URL = tunnel.public_url

print('=' * 60)
print(f'🌐 PUBLIC URL: {PUBLIC_URL}')
print('=' * 60)
print()
print('📋 Copy these lines to your local .env file:')
print()
print(f'PRODML_EMBEDDING_PROVIDER=openai')
print(f'PRODML_EMBEDDING_BASE_URL={PUBLIC_URL}/v1')
print(f'PRODML_EMBEDDING_MODEL=bge-m3')
print(f'PRODML_EMBEDDING_API_KEY=ollama')
print()
print(f'PRODML_LLM_PROVIDER=openai')
print(f'PRODML_LLM_BASE_URL={PUBLIC_URL}/v1')
print(f'PRODML_LLM_MODEL=gpt-oss:20b')
print(f'PRODML_LLM_API_KEY=ollama')

In [ ]:
# ── 6. Keep Alive (run this cell to prevent Colab from disconnecting)
# The tunnel stays active as long as this cell is running.
# Interrupt the kernel to stop.

import time
print(f'🟢 Server alive. Public URL: {PUBLIC_URL}')
print('   Press ■ Stop to terminate.')
print()

try:
    i = 0
    while True:
        i += 1
        time.sleep(60)
        print(f'   ⏱  {i} min elapsed — still running...')
except KeyboardInterrupt:
    ngrok.kill()
    print('🔴 Tunnel closed.')